# Train a model to compromise a multi-service company with TRL GRPO + LoRA

OpenRange admits **content-addressed, evolving** agent-training worlds. This notebook
trains a small model on the *cyber* pack's flagship: a **believable company** — a public
storefront in a dmz, plus internal APIs and databases on a segmented network. The flag is
a production-style credential **confined to an internal service**, with no front door. To
get it, the agent must **recon the estate, find a foothold (an SSRF on the storefront),
and pivot dmz→internal** over the network — graded by the world itself, no static dataset
of "correct" attacks.

It runs on **real Docker**: each rollout boots **one container per service** on a real
segmented network (and falls back to an in-process emulation when Docker isn't there).
Same torch-free TRL seam, same `http_get` / `submit` tools you bring — only the world
got real, and it **hardens as the agent improves**.

## 1. Install

```bash
uv pip install "openrange-trl[train]"   # torch, transformers, trl, datasets, accelerate, peft
```

`import openrange` never needs this extra — only the live training path below does. The
**CONTAINER** path needs a running Docker engine; the in-process fallback needs nothing.

## 2. Admit the company world

`company: True` builds a segmented 6–8-service estate instead of a single service — deterministic and LLM-free, so this snapshot is byte-identical on every machine. We also pick the backing: real Docker if it's running, else the in-process emulation.

In [1]:
import shutil
import subprocess

from cyber_webapp import WebappPack
from openrange_pack_sdk import Backing, Snapshot

from openrange.core.admit import admit

pack = WebappPack()
snapshot = admit(
    pack,
    manifest={
        "world": {"goal": "recover the hidden flag from the internal estate"},
        "pack": {"id": "webapp"},
        "runtime": {"tick": {"mode": "off"}},
        "npc": [],
        "seed": 3,
        "company": True,
    },
    max_repairs=3,
)
assert isinstance(snapshot, Snapshot), snapshot


def docker_ok():
    if shutil.which("docker") is None:
        return False
    try:
        probe = subprocess.run(
            ["docker", "info"], capture_output=True, timeout=10, check=False
        )
    except Exception:
        return False
    return probe.returncode == 0


BACKING = Backing.CONTAINER if docker_ok() else Backing.PROCESS
print(f"world {snapshot.snapshot_id[-12:]}   backing={BACKING.name}")

world ce3e3fcec648   backing=CONTAINER


## 3. The estate

Print the world's layout — just the graph, nothing running yet. One public **storefront** sits in the `dmz`; the other services (databases, internal APIs) are on an isolated **internal** network. The flag is a credential on one of those internal databases — so the storefront is the only way in, and the dmz→internal **pivot** is mandatory.

In [2]:
g = snapshot.graph
services = list(g.by_kind("service"))
public = next(s for s in services if s.attrs.get("exposure") == "public")
ssrf = next(v for v in g.by_kind("vulnerability") if v.attrs.get("kind") == "ssrf")
flag_host = ssrf.attrs["params"]["internal_host"]


def network_of(svc):
    edge = next(e for e in g.out_edges(svc.id, "connected_to"))
    return g.nodes[edge.dst].attrs["name"]


nets = sorted({network_of(s) for s in services})
print(f"{len(services)} services across networks {nets}\n")
for s in sorted(services, key=lambda s: s.attrs.get("exposure")):
    tag = ""
    if s.attrs.get("exposure") == "public":
        tag = "  <- public entry (the only way in)"
    elif s.attrs["name"] == flag_host:
        tag = "  <- holds the flag"
    print(
        f"  {s.attrs['name']:<16} {s.attrs.get('exposure'):<9} {network_of(s):<9}{tag}"
    )
print(f"\nflag: a hidden credential on '{flag_host}' (internal) — reachable only by")
print(f"pivoting from '{public.attrs['name']}' via the SSRF on its endpoint.")

8 services across networks ['dmz', 'internal']

  orders-db        internal  internal   <- holds the flag
  orders-api       internal  internal 
  users-db         internal  internal 
  billing-db       internal  internal 
  catalog-api      internal  internal 
  payments-api     internal  internal 
  ledger-db        internal  internal 
  storefront       public    dmz        <- public entry (the only way in)

flag: a hidden credential on 'orders-db' (internal) — reachable only by
pivoting from 'storefront' via the SSRF on its endpoint.


## 4. See the reward before you train

The reward grades the **HTTP path** against three subgoals: reach the entry, extract *something*, submit the *right* flag. Driving the env's tools by hand maps three behaviors onto it (no model, no GPU). The richest is the **full breach**: reach the storefront, **SSRF-pivot into the internal service, and exfiltrate the credential** — every coordinate read from the world graph (no hardcoded secrets) via the pack's reference solver.

In [3]:
import json
import re

from cyber_webapp.reference_solver import solve_chain
from openrange_trl import EpisodeEnv

from examples.tools import WEB_TOOLS
from openrange.core.episode import EpisodeService


def pentest_task(snap):
    return next(t for t in snap.tasks if t.meta.get("family") == "webapp.pentest")


def entry_url(snap):
    return snap.graph.nodes[pentest_task(snap).entrypoints[0]].attrs["public_url"]


def flag_from(body):
    try:
        obj = json.loads(body)
    except json.JSONDecodeError:
        obj = None
    if isinstance(obj, dict):
        for key in ("credential", "data", "flag", "secret"):
            if isinstance(obj.get(key), str):
                return obj[key]
    found = re.search(
        r"(?:ghp_|gho_|sk_live_|AKIA)[A-Za-z0-9_]+|[0-9a-f-]{32,36}", body
    )
    return found.group(0) if found else ""


def breach(env, snap):
    env.http_get(entry_url(snap))
    trace = solve_chain(snap.graph, lambda p: env.http_get(p).split(chr(10), 1)[-1])
    env.submit(json.dumps({"flag": flag_from(trace.terminal)}))


def grade(name, act):
    svc = EpisodeService(pack, f"or-runs/cyber/{name}", backing=BACKING)
    env = EpisodeEnv(
        service=svc, snapshots={snapshot.snapshot_id: snapshot}, tools=WEB_TOOLS
    )
    env.reset(snapshot_id=snapshot.snapshot_id, task_id=pentest_task(snapshot).id)
    act(env)
    env._finalize()
    print(f"{name:<26} reward={env.reward:.3f}  resolved={env.report.passed}")
    svc.close()


entry = entry_url(snapshot)
grade("reach the storefront", lambda e: e.http_get(entry))
grade(
    "reach + wrong flag",
    lambda e: (e.http_get(entry), e.submit(json.dumps({"flag": "guess"}))),
)
grade("recon + pivot + breach", lambda e: breach(e, snapshot))

reach the storefront       reward=0.333  resolved=False


reach + wrong flag         reward=0.667  resolved=False


recon + pivot + breach     reward=1.000  resolved=True


Three behaviors, three grades:

| behavior | reward | why |
|------|--------|-----|
| reach the storefront | **0.333** | `reached_endpoint` only |
| reach + wrong flag | **0.667** | `+ extracted_anything`, but wrong |
| recon + pivot + breach | **1.0** | breached: pivoted dmz→internal, exfiltrated the credential |

That `0.333 → 1.0` spread is what GRPO turns into a gradient. `matched_flag == success` — the
agent has to actually **submit the right credential**, which it can only get by pivoting,
not by poking the storefront. (This surface is asserted in `tests/test_cyber_company.py`.)

## 5. Wrap the company world as a TRL environment

The torch-free adapter exposes the three seams TRL needs — `build_grpo_dataset` (the
pentest prompt rows), `make_environment_factory` (one `EpisodeEnv` per rollout), and
`make_reward_func` (the held-out grader). We build them once here to show the wiring; §8
drives a whole evolving **pool** of worlds through these same seams.

With `backing=CONTAINER`, each rollout boots the world's **per-service containers on a
docker network**, with the storefront published on a host port the policy's `http_get`
reaches. `sandbox=False` — the in-process policy talks to the world over the wire with the
tools you bring (`http_get`, `submit`).

In [4]:
from datasets import Dataset
from openrange_trl import build_grpo_dataset, make_environment_factory, make_reward_func

num_generations = 2
rows = [
    r
    for r in build_grpo_dataset(snapshot, repeat=num_generations)
    if "pentest" in r["task_id"]
]
dataset = Dataset.from_list(rows)
environment_factory = make_environment_factory(
    pack,
    [snapshot],
    "or-runs/cyber/envs",
    tools=WEB_TOOLS,
    backing=BACKING,
    sandbox=False,
)
reward_func = make_reward_func()

## 6. Load the policy + a LoRA adapter

LoRA on a small instruct model — the base stays frozen (fits a laptop), GRPO updates only
the low-rank adapters. Bump `model_name` to a larger instruct model for rollouts strong
enough to actually land the pivot.

In [5]:
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 10703.80it/s]

## 7. Configure GRPO

`num_generations` completions of the same prompt, each against its own freshly booted
company. `beta=0` drops the reference model; `use_vllm=False` generates through
transformers; `max_tool_calling_iterations` is raised to **8** because recon → pivot →
submit is several turns, not the one-shot of a single-service bug. `max_steps=1` makes one
call here **one training round** — §8 wraps rounds into the curriculum.

In [6]:
from trl import GRPOConfig

config = GRPOConfig(
    output_dir="or-runs/cyber/trainer",
    per_device_train_batch_size=num_generations,
    num_generations=num_generations,
    steps_per_generation=1,
    gradient_accumulation_steps=1,
    max_steps=1,
    learning_rate=1e-4,
    beta=0.0,
    temperature=1.0,
    max_completion_length=256,
    max_tool_calling_iterations=8,
    use_vllm=False,
    log_completions=False,
    logging_steps=1,
    report_to="none",
    disable_tqdm=True,
    save_strategy="no",
    bf16=False,
    fp16=False,
)

## 8. Train on an evolving pool of worlds

Training runs over a *pool* of worlds, not one. Each round samples a spread from the pool — a **mix floor** always keeps an easy one in — trains a GRPO pass, then evolves the worlds the model learns most from into harder admitted children, keeping the parents. The pool is the curriculum, and it sharpens as the model improves.

`run_pool_curriculum` drives the loop; `make_grpo_rounds` returns the `(train_round, eval_round)` pair. `eval_round` scores a **fenced held-out** pool under a frozen update — no learning — so the train-vs-held-out gap measures generalization, not memorization. One real round:

In [ ]:
from cyber_webapp.difficulty import world_difficulty
from openrange_trl import make_grpo_rounds

from openrange.pool import EvalPool, WorldPool, run_pool_curriculum


def company(seed):
    return {
        "world": {"goal": "recover the hidden flag from the internal estate"},
        "pack": {"id": "webapp"},
        "runtime": {"tick": {"mode": "off"}},
        "npc": [],
        "seed": seed,
        "company": True,
    }


pool = WorldPool.seed(
    pack,
    [company(s) for s in range(3)],
    difficulty_fn=lambda s: float(world_difficulty(s.graph)),
    family="webapp.pentest",
    max_size=6,
)
held_out = EvalPool.seed(
    pack,
    [company(s) for s in (7, 8)],
    difficulty_fn=lambda s: float(world_difficulty(s.graph)),
    family="webapp.pentest",
)
train_round, eval_round = make_grpo_rounds(
    pack,
    model=model,
    args=config,
    tools=WEB_TOOLS,
    run_root="or-runs/cyber/pool",
    processing_class=tokenizer,
    peft_config=lora,
)
metrics = run_pool_curriculum(
    pool,
    train_round,
    rounds=1,
    pack=pack,
    groups=1,
    num_generations=num_generations,
    gate=lambda _snap, mut: mut.family == "webapp.pentest",
    eval_pool=held_out,
    eval_round=eval_round,
)
m = metrics[0]
print(
    f"one real GRPO round → train {m.train_solve_rate:.3f}  "
    f"held-out {m.held_out_solve_rate:.3f}  gap {m.generalization_gap:+.3f}"
)

The round trained, but the 0.5B can't pivot — its solve-rate is the **free `0.333`** the
running storefront answers before the agent acts, with zero spread, so GRPO has no gradient.
A real climb needs a bigger model on a GPU, where rollouts genuinely differ and the
*policy's* own improvement deepens the pool. To watch the **loop** itself — solve, harden,
keep an easy tail — we stand in the §4 reference breach for that competent policy below and
seed a fresh pool; swap `make_grpo_run_round` back in on a GPU and the climb drives it.

In [ ]:
def scripted_round(rows, snapshots):
    by_id = {s.snapshot_id: s for s in snapshots}
    reports = {}
    for row in rows:
        snap = by_id[row["snapshot_id"]]
        svc = EpisodeService(
            pack, f"or-runs/cyber/pool-{snap.snapshot_id[-12:]}", backing=BACKING
        )
        env = EpisodeEnv(
            service=svc, snapshots={snap.snapshot_id: snap}, tools=WEB_TOOLS
        )
        try:
            env.reset(snapshot_id=row["snapshot_id"], task_id=row["task_id"])
            breach(env, snap)
            env._finalize()
            key = (row["snapshot_id"], row["task_id"])
            reports.setdefault(key, []).append(env.report)
        finally:
            svc.close()
    return reports


pool = WorldPool.seed(
    pack,
    [company(s) for s in range(3)],
    difficulty_fn=lambda s: float(world_difficulty(s.graph)),
    family="webapp.pentest",
    max_size=6,
)
seeded = sorted(world_difficulty(s.graph) for s in pool.snapshots())
run_pool_curriculum(
    pool,
    scripted_round,
    rounds=3,
    pack=pack,
    groups=2,
    num_generations=2,
    gate=lambda _snap, mut: mut.family == "webapp.pentest",
)
band = sorted(world_difficulty(s.graph) for s in pool.snapshots())
print(
    f"pool difficulty band: seeded {seeded}  →  "
    f"after 3 rounds {band}  ({len(pool)} worlds)"
)

## 9. The pool is a population

The pool ends wider and harder than it started. Each evolved world is an **admission-checked** child: `auto_evolve` deepens or decoys the breach, re-admits it (so the harder world is still provably solvable), and records its parent. The easy seeds stay in under the mix floor. What you train on is a *band* of worlds drifting toward harder — not one fixed task — every world traceable to the one it came from.

In [ ]:
for snap in sorted(pool.snapshots(), key=lambda s: world_difficulty(s.graph)):
    evolve = snap.lineage.get("_evolve") or {}
    parent = (evolve.get("parent_snapshot_id") or "")[-12:] or "root"
    print(
        f"{snap.snapshot_id[-12:]}  difficulty={world_difficulty(snap.graph):<3} "
        f"parent={parent}"
    )

## 10. Or drive it with any agent

Nothing here is TRL-specific — the world is an HTTP target behind the `EpisodeEnv`
surface, graded the same way. Point any agent framework at the same `http_get` /
`submit` tools. Here's the company breached by a [Strands](https://strandsagents.com)
agent against any OpenAI-compatible endpoint: `pip install strands-agents openai`, set
`OPENAI_BASE_URL` (and optionally `OPENAI_MODEL`), and a capable model lands the pivot
in a handful of calls.

In [10]:
import os

endpoint = os.environ.get("OPENAI_BASE_URL")
if endpoint:
    from strands import Agent, tool
    from strands.models.openai import OpenAIModel

    live = environment_factory()
    live.reset(snapshot_id=snapshot.snapshot_id, task_id=pentest_task(snapshot).id)

    @tool
    def http_get(path: str) -> str:
        "GET a path on the target; returns the status line then the body."
        return live.http_get(path)[:1500]

    @tool
    def submit(flag: str) -> str:
        "Submit the recovered credential to end the episode."
        return live.submit(json.dumps({"flag": flag}))

    model = OpenAIModel(
        client_args={
            "base_url": endpoint,
            "api_key": os.environ.get("OPENAI_API_KEY", "EMPTY"),
        },
        model_id=os.environ.get("OPENAI_MODEL", "gpt-4o-mini"),
    )
    agent = Agent(
        model=model,
        tools=[http_get, submit],
        callback_handler=None,
        system_prompt="Recon the target, find the hidden credential, and submit it.",
    )
    agent(f"Target: {entry_url(snapshot)}")
    live._finalize()
    print(f"strands agent \u2192 reward {live.reward:.3f}")
else:
    print("set OPENAI_BASE_URL to drive the same world with any LLM agent")

set OPENAI_BASE_URL to drive the same world with any LLM agent


## Where this goes

You watched an agent breach a **real, multi-service company** — recon the estate, pivot
dmz→internal over the network, exfiltrate a confined credential — and the gym **harden** in
response. Two things take it past a laptop:

- **A real climb.** Swap the reference breach for `make_grpo_run_round` + a larger instruct
  model on a GPU. Now the *policy's* own improvement drives the hardening, no stand-in.
- **Throughput.** Each rollout boots a whole company; TRL's `AsyncGRPOTrainer` pipelines many
  against a vLLM server with the same `make_environment_factory`. Both trainers reset worlds
  serially, so a pooled / fast world realization is the gym-side lever that keeps a big batch
  from stalling on container boots (#294).

Honest scope: a laptop-scale model doesn't complete the pivot — the reward surface and the
breach are real and proven (§4), but mastery needs GPU-scale compute. The deterministic
world + reward tests live in `packs/cyber_webapp` + `tests/test_cyber_company.py`.